In [28]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("demo").getOrCreate()

In [74]:
data = [{"id":1, "full_name":"Surya Pratap Singh", "age": 20},
        {"id":2, "full_name":"Surya P Singh", "age": 30},
        {"id":3, "full_name":"Surya Singh", "age": 40}]

In [75]:
df = spark.createDataFrame(data)

In [76]:
df.show()

+---+------------------+---+
|age|         full_name| id|
+---+------------------+---+
| 20|Surya Pratap Singh|  1|
| 30|     Surya P Singh|  2|
| 40|       Surya Singh|  3|
+---+------------------+---+



## Get first name and last name

In [77]:
from pyspark.sql.functions import col, split, slice, element_at, concat_ws

In [78]:
df.withColumn('first_name1',split(col('full_name'), ' ')[0])\
    .withColumn('first_name',
                concat_ws(' ',
                          slice(split(col('full_name'), ' '), 
                                1, 
                                size(split(df['full_name'], ' ')) - 1)))\
    .withColumn('last_name',split(col('full_name'), ' ')[size(split(df['full_name'], ' ')) - 1])\
    .withColumn('last_name1',element_at(split(col('full_name'), ' '), -1)).show()

+---+------------------+---+-----------+------------+---------+----------+
|age|         full_name| id|first_name1|  first_name|last_name|last_name1|
+---+------------------+---+-----------+------------+---------+----------+
| 20|Surya Pratap Singh|  1|      Surya|Surya Pratap|    Singh|     Singh|
| 30|     Surya P Singh|  2|      Surya|     Surya P|    Singh|     Singh|
| 40|       Surya Singh|  3|      Surya|       Surya|    Singh|     Singh|
+---+------------------+---+-----------+------------+---------+----------+



## Create masked email address

In [79]:
data = [
    (1, 'surya.singh@gmail.com'),
    (2, 'surya@company.com'),
    (3, 'surya.p.singh@yahoo.com'),
    (4, None),
    (5, 'ss@short.com')
]

df = spark.createDataFrame(data, ['user_id', 'email'])
df.show(truncate=False)

+-------+-----------------------+
|user_id|email                  |
+-------+-----------------------+
|1      |surya.singh@gmail.com  |
|2      |surya@company.com      |
|3      |surya.p.singh@yahoo.com|
|4      |NULL                   |
|5      |ss@short.com           |
+-------+-----------------------+



In [80]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

#mask_email_udf = udf(mask_email, StringType())

@udf(StringType())
def mask_email(email):
    if email is None:
        return None

    try:
        username, domain = email.split('@')
        if len(username) <= 2:
            masked = "*" * len(username)
        else:
            masked = username[0] + '*' * (len(username) - 2) + username[-1]

        return masked + '@' + domain
    except Exception:
        return None

In [82]:
df.withColumn('masked_email', mask_email(col('email'))).show()

+-------+--------------------+--------------------+
|user_id|               email|        masked_email|
+-------+--------------------+--------------------+
|      1|surya.singh@gmail...|s*********h@gmail...|
|      2|   surya@company.com|   s***a@company.com|
|      3|surya.p.singh@yah...|s***********h@yah...|
|      4|                NULL|                NULL|
|      5|        ss@short.com|        **@short.com|
+-------+--------------------+--------------------+

